In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -U bitsandbytes>=0.46.1

In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# libraries
import re, os, gc, torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [ ]:
# paths

BASE = "/content/drive/MyDrive/..."          # This paths need to be updated
TRANS_DIR = f"{BASE}/translations"
QWEN_DIR = f"{BASE}/results/qwen3guard"
os.makedirs(QWEN_DIR, exist_ok=True)


# languages

LANGUAGES = {
    "english": {}, # original
    "turkish": {},
    "spanish": {},
    "asturian": {},
}

# models & config

QWEN_MODEL = "Qwen/Qwen3Guard-Gen-4B"
MAX_NEW_TOKENS = 128
CHECKPOINT_N = 50    # save every 50 prompts


In [ ]:
# load translations from CSV

translations = {}
for lang_name in LANGUAGES:
    path = f"{TRANS_DIR}/{lang_name}.csv"
    if os.path.exists(path):
        translations[lang_name] = pd.read_csv(path)
        print(f"[{lang_name}] loaded({len(translations[lang_name])} rows)")
    else:
        print(f"[{lang_name}] not found at {path}")

In [ ]:
# load model Qwen3Guard-4B

# first empty cache
torch.cuda.empty_cache()
gc.collect()

# then quantization & load
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

qwen_tok = AutoTokenizer.from_pretrained(QWEN_MODEL)
qwen_mdl = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL, quantization_config=quant, device_map="auto"
)

print("Model ready!")

In [ ]:
# functions
def extract_label_and_categories(content):
    safe_match = re.search(r"Safety: (Safe|Unsafe|Controversial)", content)
    label = safe_match.group(1) if safe_match else None
    categories = re.findall(
        r"(Violent|Non-violent Illegal Acts|Sexual Content or Sexual Acts|"
        r"PII|Suicide & Self-Harm|Unethical Acts|Politically Sensitive Topics|"
        r"Copyright Violation|Jailbreak|None)",
        content
    )
    return label, categories


def classify_sentence(sentence):
    messages = [{"role": "user", "content": sentence}]
    text = qwen_tok.apply_chat_template(messages, tokenize=False)
    inputs = qwen_tok([text], return_tensors="pt").to(qwen_mdl.device)
    generated = qwen_mdl.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    output_ids = generated[0][len(inputs.input_ids[0]):].tolist()
    content = qwen_tok.decode(output_ids, skip_special_tokens=True)
    label, cats = extract_label_and_categories(content)
    return label, cats, content


def classify_language(lang_name, trans_df):
    out_path = f"{QWEN_DIR}/{lang_name}.csv"

    # checkpoints
    if os.path.exists(out_path):
        done_df = pd.read_csv(out_path)
        results = done_df.to_dict("records")
        done_idxs = set(done_df["original_idx"].tolist())
        print(f"[{lang_name}] Loading from checkpoint ({len(done_df)} done)")
    else:
        results, done_idxs = [], set()

    pending = trans_df[~trans_df["original_idx"].isin(done_idxs)]

    if pending.empty:
        print(f"[{lang_name}] ready!")
        return

    print(f"[{lang_name}] Classifying {len(pending)} prompts...")

    for i, (_, row) in enumerate(tqdm(pending.iterrows(), total=len(pending), desc=lang_name)):
        try:
            label, cats, raw_output = classify_sentence(row["sentence_trans"])
        except Exception as e:
            label, cats, raw_output = None, [], f"ERROR: {e}"

        results.append({
            "original_idx": row["original_idx"],
            "sentence_en": row["sentence_en"],
            "sentence_trans": row["sentence_trans"],
            "language": lang_name,
            "harmful": row["harmful"], # added harmful column
            "type": row["type"], # added type column
            "predicted_safety": label,
            "predicted_categories": cats,
            "raw_output": raw_output,
        })

        if i % CHECKPOINT_N == 0 and i > 0:
            pd.DataFrame(results).to_csv(out_path, index=False)

    pd.DataFrame(results).to_csv(out_path, index=False)
    print(f"[{lang_name}] Saved at {out_path}")

In [ ]:
# classification

for lang_name, trans_df in translations.items():
    print(f"\n{'='*50}")
    classify_language(lang_name, trans_df)

print("\nClassification done!")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_results(lang_name):
    path = f"{QWEN_DIR}/{lang_name}.csv"
    if not os.path.exists(path):
        print(f"[{lang_name}] not found")
        return

    df = pd.read_csv(path)

    # Map predicted_safety to binary (1=unsafe, 0=safe)
    df["predicted_binary"] = df["predicted_safety"].apply(
        lambda x: 1 if str(x).lower() == "unsafe" else 0
    )

    # Drop rows where prediction failed
    df = df.dropna(subset=["predicted_safety"])

    print(f"\n{'='*50}")
    print(f"{lang_name.upper()} — {len(df)} prompts")
    print(f"{'='*50}")

    print("\nClassification Report:")
    print(classification_report(
        df["harmful"],
        df["predicted_binary"],
        target_names=["safe", "unsafe"]
    ))

    print("Confusion Matrix:")
    cm = confusion_matrix(df["harmful"], df["predicted_binary"])
    cm_df = pd.DataFrame(cm,
        index=["actual_safe", "actual_unsafe"],
        columns=["pred_safe", "pred_unsafe"]
    )
    print(cm_df)

    # Overall accuracy
    acc = (df["harmful"] == df["predicted_binary"]).mean()
    print(f"\nAccuracy: {acc:.2%}")

# Run for all languages
for lang_name in LANGUAGES:
    evaluate_results(lang_name)


ENGLISH — 1120 prompts

Classification Report:
              precision    recall  f1-score   support

        safe       0.99      0.99      0.99       600
      unsafe       0.99      0.99      0.99       520

    accuracy                           0.99      1120
   macro avg       0.99      0.99      0.99      1120
weighted avg       0.99      0.99      0.99      1120

Confusion Matrix:
               pred_safe  pred_unsafe
actual_safe          597            3
actual_unsafe          6          514

Accuracy: 99.20%

TURKISH — 1120 prompts

Classification Report:
              precision    recall  f1-score   support

        safe       0.95      0.98      0.97       600
      unsafe       0.98      0.94      0.96       520

    accuracy                           0.96      1120
   macro avg       0.97      0.96      0.96      1120
weighted avg       0.96      0.96      0.96      1120

Confusion Matrix:
               pred_safe  pred_unsafe
actual_safe          590           10
actual